In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
patients_data = [
    ("P101", "Rahul Sharma", "Hyderabad", "Telangana", 35, "Male", "Active"),
    ("P102", "Priya Reddy", "Bangalore", "Karnataka", 29, "Female", "Active"),
    ("P103", "Amit Kumar", "Mumbai", "Maharashtra", 42, "Male", "Inactive"),
    ("P104", "Sneha Patel", "Delhi", "Delhi", 31, "Female", "Active"),
    ("P105", "Farhan Ali", "Chennai", "Tamil Nadu", 55, "Male", "Active"),
    ("P106", "Neha Singh", "Pune", "Maharashtra", 38, "Female", "Inactive"),
    ("P107", "Arjun Verma", "Hyderabad", "Telangana", 26, "Male", "Active"),
    ("P108", "Meera Nair", "Kochi", "Kerala", 48, "Female", "Active")
]
df_patients_raw = spark.createDataFrame(patients_data, ["patient_id", "patient_name", "city", "state", "age", "gender", "insurance_status"])

In [0]:
doctors_data = [
    ("D101", "Dr. Ramesh", "Cardiology", "Hyderabad", 1500),
    ("D102", "Dr. Priya", "Neurology", "Bangalore", 2000),
    ("D103", "Dr. Anita", "Dermatology", "Chennai", 1000),
    ("D104", "Dr. Suresh", "Orthopedics", "Mumbai", 2500),
    ("D105", "Dr. Meera", "Pediatrics", "Delhi", 1200),
    ("D106", "Dr. Kiran", "Cardiology", "Hyderabad", 3000)
]
df_doctors_raw = spark.createDataFrame(doctors_data, ["doctor_id", "doctor_name", "department", "city", "consultation_fee"])

In [0]:
appointments_data = [
    ("A1001", "P101", "D101", "2026-06-01", "Heart Checkup", 5000, "Completed"),
    ("A1002", "P102", "D102", "2026-06-01", "Migraine", 3500, "Completed"),
    ("A1003", "P103", "D103", "2026-06-02", "Skin Allergy", 2000, "Pending"),
    ("A1004", "P104", "D104", "2026-06-02", "Fracture", 12000, "Completed"),
    ("A1005", "P105", "D105", "2026-06-03", "Fever", 1500, "Completed"),
    ("A1006", "P106", "D106", "2026-06-03", "Heart Checkup", 7000, "Completed"),
    ("A1007", "P107", "D101", "2026-06-04", "Chest Pain", 5500, "Completed"),
    ("A1008", "P108", "D103", "2026-06-04", "Skin Infection", 2500, "Pending"),
    ("A1009", "P101", "D106", "2026-06-05", "Cardiac Review", 6500, "Completed"),
    ("A1010", "P104", "D104", "2026-06-05", "Back Pain", 4500, "Cancelled")
]
df_appointments_raw = spark.createDataFrame(appointments_data, ["appointment_id", "patient_id", "doctor_id", "appointment_date", "diagnosis", "bill_amount", "status"])

In [0]:
json_string = """[
    {"patient_id": "P101", "preferred_hospital": "Apollo Hospital", "contact": {"phone": "9876500011", "email": "rahul@mail.com"}},
    {"patient_id": "P102", "preferred_hospital": "Yashoda Hospital", "contact": {"phone": "9876500012", "email": "priya@mail.com"}},
    {"patient_id": "P104", "preferred_hospital": "Care Hospital", "contact": {"phone": "9876500014", "email": "sneha@mail.com"}},
    {"patient_id": "P108", "preferred_hospital": "Apollo Hospital", "contact": {"phone": "9876500018", "email": "meera@mail.com"}}
]"""

# 1. Create a simple 1-row DataFrame containing the raw JSON string
df_raw_string = spark.createDataFrame([(json_string,)], ["raw_json"])

In [0]:
json_schema = StructType([
    StructField("patient_id", StringType(), True),
    StructField("preferred_hospital", StringType(), True),
    StructField("contact", StructType([
        StructField("phone", StringType(), True),
        StructField("email", StringType(), True)
    ]), True)
])

In [0]:
inline_data = [("""[
    {"patient_id": "P101", "preferred_hospital": "Apollo Hospital", "contact": {"phone": "9876500011", "email": "rahul@mail.com"}},
    {"patient_id": "P102", "preferred_hospital": "Yashoda Hospital", "contact": {"phone": "9876500012", "email": "priya@mail.com"}},
    {"patient_id": "P104", "preferred_hospital": "Care Hospital", "contact": {"phone": "9876500014", "email": "sneha@mail.com"}},
    {"patient_id": "P108", "preferred_hospital": "Apollo Hospital", "contact": {"phone": "9876500018", "email": "meera@mail.com"}}
]""",)]
df_raw_json = spark.createDataFrame(inline_data, ["raw_text"])

In [0]:
df_preferences_raw = df_raw_json \
    .select(F.explode(F.from_json(F.col("raw_text"), F.concat(F.lit("array<"), F.lit(json_schema.simpleString()), F.lit(">")))).alias("parsed_data")) \
    .select("parsed_data.*")

print("Patient Preferences parsed flawlessly via pure Serverless Column logic!")

Patient Preferences parsed flawlessly via pure Serverless Column logic!


In [0]:
print("\n--- Part 1: Displaying Bronze Schemas ---")
df_patients_raw.printSchema()
df_doctors_raw.printSchema()
df_appointments_raw.printSchema()
df_preferences_raw.printSchema()


--- Part 1: Displaying Bronze Schemas ---
root
 |-- patient_id: string (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- insurance_status: string (nullable = true)

root
 |-- doctor_id: string (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- city: string (nullable = true)
 |-- consultation_fee: long (nullable = true)

root
 |-- appointment_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- appointment_date: string (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- bill_amount: long (nullable = true)
 |-- status: string (nullable = true)

root
 |-- patient_id: string (nullable = true)
 |-- preferred_hospital: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- phone: str

In [0]:
appointments_data = [
    ("A1001", "P101", "D101", "2026-06-01", "Heart Checkup", 5000, "Completed"),
    ("A1002", "P102", "D102", "2026-06-01", "Migraine", 3500, "Completed"),
    ("A1003", "P103", "D103", "2026-06-02", "Skin Allergy", 2000, "Pending"),
    ("A1004", "P104", "D104", "2026-06-02", "Fracture", 12000, "Completed"),
    ("A1005", "P105", "D105", "2026-06-03", "Fever", 1500, "Completed"),
    ("A1006", "P106", "D106", "2026-06-03", "Heart Checkup", 7000, "Completed"),
    ("A1007", "P107", "D101", "2026-06-04", "Chest Pain", 5500, "Completed"),
    ("A1008", "P108", "D103", "2026-06-04", "Skin Infection", 2500, "Pending"),
    ("A1009", "P101", "D106", "2026-06-05", "Cardiac Review", 6500, "Completed"),
    ("A1010", "P104", "D104", "2026-06-05", "Back Pain", 4500, "Cancelled")
]
df_appointments_raw = spark.createDataFrame(appointments_data, ["appointment_id", "patient_id", "doctor_id", "appointment_date", "diagnosis", "bill_amount", "status"])

In [0]:
df_patients_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_patients")
df_doctors_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_doctors")
df_appointments_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_appointments")
df_preferences_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_patient_preferences")

In [0]:
print("\n--- Part 2: Generating Silver Enriched Tables ---")


--- Part 2: Generating Silver Enriched Tables ---


In [0]:
df_preferences_flat = df_preferences_raw.select(
    F.col("patient_id").alias("pref_patient_id"),
    F.col("preferred_hospital"),
    F.col("contact.phone").alias("patient_phone"),
    F.col("contact.email").alias("patient_email")
).fillna({"preferred_hospital": "Unknown"})

In [0]:
df_patients_enriched = df_patients_raw.join(df_preferences_flat, df_patients_raw.patient_id == df_preferences_flat.pref_patient_id, "left").drop("pref_patient_id")

df_silver_pipeline = df_appointments_raw \
    .join(df_patients_enriched, "patient_id", "inner") \
    .join(df_doctors_raw.select("doctor_id", "doctor_name", "department", "consultation_fee"), "doctor_id", "inner")

In [0]:
df_silver_transformed = df_silver_pipeline \
    .withColumn("final_bill", F.col("bill_amount") + F.col("consultation_fee")) \
    .withColumn("appointment_month", F.date_format(F.to_date("appointment_date", "yyyy-MM-dd"), "MMMM")) \
    .withColumn("patient_age_group", 
                F.when(F.col("age") >= 50, "Senior")
                 .when(F.col("age") >= 30, "Adult")
                 .otherwise("Young"))

In [0]:
df_silver_transformed.write.format("delta").mode("overwrite").saveAsTable("silver_healthcare_master")
print("Silver Layer Created Successfully.")

Silver Layer Created Successfully.


In [0]:
df_silver_transformed.createOrReplaceTempView("v_silver_healthcare")

In [0]:
spark.sql("SELECT SUM(final_bill) AS total_hospital_revenue FROM v_silver_healthcare").show()

+----------------------+
|total_hospital_revenue|
+----------------------+
|                 69200|
+----------------------+



In [0]:
spark.sql("SELECT department, SUM(final_bill) AS department_revenue FROM v_silver_healthcare GROUP BY department ORDER BY department_revenue DESC").show()

+-----------+------------------+
| department|department_revenue|
+-----------+------------------+
| Cardiology|             33000|
|Orthopedics|             21500|
|Dermatology|              6500|
|  Neurology|              5500|
| Pediatrics|              2700|
+-----------+------------------+



In [0]:
spark.sql("SELECT city, SUM(final_bill) AS city_revenue FROM v_silver_healthcare GROUP BY city ORDER BY city_revenue DESC").show()

+---------+------------+
|     city|city_revenue|
+---------+------------+
|Hyderabad|       23000|
|    Delhi|       21500|
|     Pune|       10000|
|Bangalore|        5500|
|    Kochi|        3500|
|   Mumbai|        3000|
|  Chennai|        2700|
+---------+------------+



In [0]:
spark.sql("SELECT * FROM v_silver_healthcare WHERE status = 'Completed'").show()

+---------+----------+--------------+----------------+--------------+-----------+---------+------------+---------+-----------+---+------+----------------+------------------+-------------+--------------+-----------+-----------+----------------+----------+-----------------+-----------------+
|doctor_id|patient_id|appointment_id|appointment_date|     diagnosis|bill_amount|   status|patient_name|     city|      state|age|gender|insurance_status|preferred_hospital|patient_phone| patient_email|doctor_name| department|consultation_fee|final_bill|appointment_month|patient_age_group|
+---------+----------+--------------+----------------+--------------+-----------+---------+------------+---------+-----------+---+------+----------------+------------------+-------------+--------------+-----------+-----------+----------------+----------+-----------------+-----------------+
|     D106|      P101|         A1009|      2026-06-05|Cardiac Review|       6500|Completed|Rahul Sharma|Hyderabad|  Telangana| 

In [0]:
spark.sql("SELECT patient_id, patient_name, SUM(final_bill) AS total_patient_billing FROM v_silver_healthcare GROUP BY patient_id, patient_name ORDER BY total_patient_billing DESC").show()

+----------+------------+---------------------+
|patient_id|patient_name|total_patient_billing|
+----------+------------+---------------------+
|      P104| Sneha Patel|                21500|
|      P101|Rahul Sharma|                16000|
|      P106|  Neha Singh|                10000|
|      P107| Arjun Verma|                 7000|
|      P102| Priya Reddy|                 5500|
|      P108|  Meera Nair|                 3500|
|      P103|  Amit Kumar|                 3000|
|      P105|  Farhan Ali|                 2700|
+----------+------------+---------------------+



In [0]:
win_doc_rev = Window.partitionBy("doctor_id", "doctor_name").orderBy(F.col("final_bill").desc())
win_dept = Window.orderBy(F.desc("total_revenue"))
win_patient = Window.orderBy(F.desc("total_patient_billing"))
win_dept_doc = Window.partitionBy("department").orderBy(F.desc("total_doctor_revenue"))
win_running = Window.orderBy("appointment_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

In [0]:
df_doc_revenue = df_silver_transformed.groupBy("doctor_id", "doctor_name").agg(F.sum("final_bill").alias("total_revenue"))
df_doc_revenue.withColumn("rank", F.dense_rank().over(Window.orderBy(F.desc("total_revenue")))).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+-------------+----+
|doctor_id|doctor_name|total_revenue|rank|
+---------+-----------+-------------+----+
|     D104| Dr. Suresh|        21500|   1|
|     D106|  Dr. Kiran|        19500|   2|
|     D101| Dr. Ramesh|        13500|   3|
|     D103|  Dr. Anita|         6500|   4|
|     D102|  Dr. Priya|         5500|   5|
|     D105|  Dr. Meera|         2700|   6|
+---------+-----------+-------------+----+



In [0]:
df_dept_revenue = df_silver_transformed.groupBy("department").agg(F.sum("final_bill").alias("total_revenue"))
df_dept_revenue.withColumn("rank", F.dense_rank().over(win_dept)).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------------+----+
| department|total_revenue|rank|
+-----------+-------------+----+
| Cardiology|        33000|   1|
|Orthopedics|        21500|   2|
|Dermatology|         6500|   3|
|  Neurology|         5500|   4|
| Pediatrics|         2700|   5|
+-----------+-------------+----+



In [0]:
df_patient_billing = df_silver_transformed.groupBy("patient_id", "patient_name").agg(F.sum("final_bill").alias("total_patient_billing"))
df_patient_billing.withColumn("rank", F.dense_rank().over(win_patient)).filter("rank <= 3").show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+------------+---------------------+----+
|patient_id|patient_name|total_patient_billing|rank|
+----------+------------+---------------------+----+
|      P104| Sneha Patel|                21500|   1|
|      P101|Rahul Sharma|                16000|   2|
|      P106|  Neha Singh|                10000|   3|
+----------+------------+---------------------+----+



In [0]:
df_dept_doc_revenue = df_silver_transformed.groupBy("department", "doctor_id", "doctor_name").agg(F.sum("final_bill").alias("total_doctor_revenue"))
df_dept_doc_revenue.withColumn("rank", F.dense_rank().over(win_dept_doc)).filter("rank == 1").show()

+-----------+---------+-----------+--------------------+----+
| department|doctor_id|doctor_name|total_doctor_revenue|rank|
+-----------+---------+-----------+--------------------+----+
| Cardiology|     D106|  Dr. Kiran|               19500|   1|
|Dermatology|     D103|  Dr. Anita|                6500|   1|
|  Neurology|     D102|  Dr. Priya|                5500|   1|
|Orthopedics|     D104| Dr. Suresh|               21500|   1|
| Pediatrics|     D105|  Dr. Meera|                2700|   1|
+-----------+---------+-----------+--------------------+----+



In [0]:
df_date_revenue = df_silver_transformed.groupBy("appointment_date").agg(F.sum("final_bill").alias("daily_revenue"))
df_date_revenue.withColumn("running_revenue_total", F.sum("daily_revenue").over(win_running)).orderBy("appointment_date").show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------------+-------------+---------------------+
|appointment_date|daily_revenue|running_revenue_total|
+----------------+-------------+---------------------+
|      2026-06-01|        12000|                12000|
|      2026-06-02|        17500|                29500|
|      2026-06-03|        12700|                42200|
|      2026-06-04|        10500|                52700|
|      2026-06-05|        16500|                69200|
+----------------+-------------+---------------------+



In [0]:
df_silver_transformed.write.format("delta").mode("overwrite").save("/mnt/healthcare/delta_table")

In [0]:
df_silver_transformed.write.format("delta").mode("overwrite").saveAsTable("delta_healthcare_managed")

In [0]:
spark.sql("CREATE TABLE IF NOT EXISTS delta_healthcare_sql USING DELTA AS SELECT * FROM v_silver_healthcare")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
print("Reviewing Storage History Version Logs...")
display(spark.sql("DESCRIBE HISTORY delta_healthcare_managed"))

Reviewing Storage History Version Logs...


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-22T05:01:29.000Z,147985412235563,azuser7213_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2779838936692752),dad97603-5d79-480b-8e29-13c7075b7d20,0622-045037-5uhgjjc9-v2n,null,WriteSerializable,false,"Map(numFiles -> 8, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 10, numOutputBytes -> 44367)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
df_version_zero = spark.read.format("delta").option("versionAsOf", 0).table("delta_healthcare_managed")

In [0]:
update_records = [("P102", "Priya Reddy", "Pune", "Maharashtra", 29, "Female", "Inactive")]
df_updates = spark.createDataFrame(update_records, ["patient_id", "patient_name", "city", "state", "age", "gender", "insurance_status"])
df_updates.createOrReplaceTempView("v_incoming_updates")

print("Executing SCD Type 1 Overwrite Merge Routine...")
spark.sql("""
    MERGE INTO delta_healthcare_managed AS target
    USING (
        SELECT u.patient_id, u.city, u.insurance_status 
        FROM v_incoming_updates u
    ) AS source
    ON target.patient_id = source.patient_id
    WHEN MATCHED THEN
        UPDATE SET target.city = source.city, target.insurance_status = source.insurance_status
""")

Executing SCD Type 1 Overwrite Merge Routine...


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
print("Verifying Incremented Transaction Logs...")
display(spark.sql("DESCRIBE HISTORY delta_healthcare_managed"))

Verifying Incremented Transaction Logs...


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-06-22T05:02:11.000Z,147985412235563,azuser7213_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(patient_id#17858 = patient_id#17827)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(2779838936692752),51acd449-3535-4f97-a2d8-476935e313c6,0622-045037-5uhgjjc9-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 5565, numTargetBytesRemoved -> 5574, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4498, materializeSourceTimeMs -> 335, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2570, numTargetRowsUpdated -> 1, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1508)",null,Databricks-Runtime/18.2.x-photon-scala2.13
0,2026-06-22T05:01:29.000Z,147985412235563,azuser7213_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2779838936692752),dad97603-5d79-480b-8e29-13c7075b7d20,0622-045037-5uhgjjc9-v2n,null,WriteSerializable,false,"Map(numFiles -> 8, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 10, numOutputBytes -> 44367)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
# 1. Run your storage layouts optimization
spark.sql("OPTIMIZE delta_healthcare_managed ZORDER BY (patient_id)")

# 2. Run VACUUM directly (Serverless handles parallel execution automatically!)
spark.sql("VACUUM delta_healthcare_managed RETAIN 168 HOURS")

print("Delta optimizations complete on Serverless compute!")

Delta optimizations complete on Serverless compute!


In [0]:
%sql
SELECT department, SUM(final_bill) AS total_revenue FROM delta_healthcare_managed GROUP BY department;

department,total_revenue
Cardiology,33000
Neurology,5500
Dermatology,6500
Pediatrics,2700
Orthopedics,21500


In [0]:
%sql
SELECT city, SUM(final_bill) AS total_revenue FROM delta_healthcare_managed GROUP BY city;

city,total_revenue
Hyderabad,23000
Pune,15500
Mumbai,3000
Chennai,2700
Delhi,21500
Kochi,3500


In [0]:
%sql
SELECT status, COUNT(*) AS appointment_count FROM delta_healthcare_managed GROUP BY status;

status,appointment_count
Completed,7
Pending,2
Cancelled,1


In [0]:
%sql
SELECT 
    doctor_name, 
    SUM(final_bill) AS total_revenue 
FROM delta_healthcare_managed 
GROUP BY doctor_name 
ORDER BY total_revenue DESC;

doctor_name,total_revenue
Dr. Suresh,21500
Dr. Kiran,19500
Dr. Ramesh,13500
Dr. Anita,6500
Dr. Priya,5500
Dr. Meera,2700


In [0]:
%sql
SELECT appointment_date, SUM(final_bill) AS daily_revenue FROM delta_healthcare_managed GROUP BY appointment_date ORDER BY appointment_date;

appointment_date,daily_revenue
2026-06-01,12000
2026-06-02,17500
2026-06-03,12700
2026-06-04,10500
2026-06-05,16500


In [0]:
df_silver_transformed.write.format("delta").mode("overwrite").saveAsTable("managed_healthcare_table")

In [0]:
# Clear out the path option so Unity Catalog automatically stores it safely
(df_silver_transformed.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("external_healthcare_table_validated"))
    

In [0]:
df_silver_transformed.createOrReplaceTempView("temp_view_healthcare")

In [0]:
# Create a permanent view directly from your persistent Silver Delta Table asset
spark.sql("""
    CREATE OR REPLACE VIEW global_shared_healthcare_view 
    AS SELECT * FROM silver_healthcare_master
""")

print("Tables and Views built successfully under Serverless constraints.")

Tables and Views built successfully under Serverless constraints.
